In [ ]:
import os
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ---------------- 설정 ----------------
# 'asof'  : merge_asof backward. 2025 test 행이 가장 최근(2024) 트랙맨 값을 받는다.
#           학습/추론이 동일한 규칙을 쓰므로 원칙적으로 더 타당하다.
# 'exact' : (season, month) 정확 일치 + fillna(0). 900점 버전과 완전히 동일한 동작
#           (트랙맨에 2025가 없어 test에서는 전부 0이 된다).
TRACKMAN_MODE = 'asof'

N_SPLITS = 10                 # 5 -> 10 (각 fold가 90%를 학습, 평균 대상도 늘어 분산 감소)
SEEDS = [42, 202, 2024]       # seed 앙상블 (3개) -> 총 N_SPLITS * len(SEEDS) = 30개 모델
N_OPTUNA_TRIALS = 40          # 하이퍼파라미터 탐색 횟수

# --- 2026-08-18 추가 (2024 시즌 홀드아웃 3-seed 짝지어 검증 결과 반영) ---
# 조건부 투수통계: 기준선 대비 +20~27점 (3 seed 전부 우세, 분산도 ±18->±6로 감소)
USE_COND_STATS = True
# 재중심화: 11개 설정 전부에서 +11~22점 (평균 +18)
RECENTER = True
HOLDOUT_SEASON = 2024         # 오프셋 측정용 홀드아웃 시즌 (이 시즌은 학습에서 빼고 1회 측정)
N_HOLDOUT_FOLDS = 3
# 죽은 피처: asof_pitcher_n이 '경기내'가 아니라 '커리어 누적'이라 의도대로 동작하지 않음
#   is_long_relief 는 전체의 86%(이닝>1 중 97%)로 사실상 inning>1 과 동일,
#   is_strict_inherited_runner 는 0.05%로 상수, pitches_per_inning 은 커리어투구수/이닝.
#   (효과는 +4점 수준으로 미미하나 코드 정합성 차원에서 제거)
DEAD_FEATURES = ['is_long_relief', 'is_short_relief',
                 'is_strict_inherited_runner', 'pitches_per_inning']
print(f"TRACKMAN_MODE = {TRACKMAN_MODE} | N_SPLITS = {N_SPLITS} | SEEDS = {SEEDS}")

# v5: Optuna 재탐색을 끈다. 다시 탐색하면 파라미터가 바뀌어 리더보드 차이가
# '트랙맨 v2 효과'인지 '파라미터 변화'인지 구분되지 않는다 (v4 때 실제로 겪음).
# 아래는 v4(987.3936) 실행에서 나온 값 그대로.
RUN_OPTUNA = False
V4_BEST_PARAMS = {
    "learning_rate": 0.022831883708228414,
    "depth": 8,
    "l2_leaf_reg": 8.552069332567962,
    "bagging_temperature": 0.05636104060100738,
    "random_strength": 0.7731135614050382
}

# v6(릴리스 동역학 12개)는 리더보드 988.4720 으로 v5(990.9528) 대비 -2.48 -> 기각.
# 코드는 보존하되 기본 False. 스크리닝 +20 / 누수없는 홀드아웃 +5 / 실측 -2.48 이었다.
USE_RELEASE_DYNAMICS = False


In [ ]:
STEPS_SRC = r"""
def step1_basic_features(df):
    df_proc = df.copy()
    df_proc['is_weekend_day_game'] = np.where(
        (df_proc['game_month'].isin([4, 5, 9, 10])) & (df_proc['game_dayofweek'].isin([5, 6])), 1.0, 0.0)
    df_proc['is_heat_wave_game'] = np.where(df_proc['game_month'].isin([7, 8]), 1.0, 0.0)
    return df_proc


def step2_pitcher_role_features(df):
    df_proc = df.copy()
    df_proc['is_pure_starter'] = np.where(df_proc['inning'] == 1, 1.0, 0.0)
    df_proc['is_long_relief'] = np.where(
        (df_proc['inning'] > 1) & (df_proc['asof_pitcher_n'] >= (df_proc['inning'] - 1) * 12), 1.0, 0.0)
    df_proc['is_short_relief'] = np.where(
        (df_proc['inning'] > 1) & (df_proc['asof_pitcher_n'] < (df_proc['inning'] - 1) * 12), 1.0, 0.0)
    return df_proc


def step3_matchup_features(df):
    df_proc = df.copy()
    if 'pitcher_hand' in df_proc.columns and 'batter_hand' in df_proc.columns:
        df_proc['is_same_hand'] = np.where(df_proc['pitcher_hand'] == df_proc['batter_hand'], 1.0, 0.0)
    return df_proc


def step4_refined_count_features(df):
    df_proc = df.copy()
    b, s = df_proc['balls_before'], df_proc['strikes_before']
    df_proc['is_first_pitch'] = np.where((b == 0) & (s == 0), 1.0, 0.0)
    df_proc['is_full_count'] = np.where((b == 3) & (s == 2), 1.0, 0.0)
    pitcher_ahead = ((b == 0) & (s == 1)) | ((b == 0) & (s == 2)) | ((b == 1) & (s == 2))
    batter_ahead = ((b == 1) & (s == 0)) | ((b == 2) & (s == 0)) | ((b == 3) & (s == 0)) | ((b == 2) & (s == 1)) | ((b == 3) & (s == 1))
    neutral = ((b == 1) & (s == 1)) | ((b == 2) & (s == 2))
    df_proc['count_advantage'] = np.select(
        [pitcher_ahead, batter_ahead, neutral], ['Pitcher', 'Batter', 'Neutral'], default='None')
    df_proc['is_waste_pitch_sit'] = np.where(((b == 0) & (s == 2)) | ((b == 1) & (s == 2)), 1.0, 0.0)
    df_proc['is_must_strike_sit'] = np.where(((b == 3) & (s == 0)) | ((b == 3) & (s == 1)), 1.0, 0.0)
    return df_proc


def step5_pitches_per_inning(df):
    df_proc = df.copy()
    df_proc['pitches_per_inning'] = df_proc['asof_pitcher_n'] / df_proc['inning'].clip(lower=1)
    return df_proc


def step6_combined_runner_features(df):
    df_proc = df.copy()
    df_proc['is_risp'] = df_proc['base_state'].astype(str).apply(
        lambda x: 1.0 if ('2' in x) or ('3' in x) else 0.0)
    df_proc['is_strict_inherited_runner'] = np.where(
        (df_proc['inning'] > 1) & (df_proc['asof_pitcher_n'] < 5) & (df_proc['num_runners_on'] > 0), 1.0, 0.0)
    df_proc['is_self_risp'] = np.where(
        (df_proc['asof_pitcher_n'] >= 15) & (df_proc['is_risp'] == 1.0), 1.0, 0.0)
    li_filled = df_proc['li'].fillna(0)
    df_proc['risp_pressure_index'] = df_proc['is_risp'] * li_filled
    df_proc['is_steal_threat_sit'] = np.where(
        (df_proc['runner_on_1b'] == 1) & (df_proc['runner_on_2b'] == 0)
        & (df_proc['score_diff_pitcher_team'].abs() <= 3), 1.0, 0.0)
    return df_proc


def step7_bayesian_smoothing(df, prior_mean=0.64):
    df_proc = df.copy()
    C = 50
    if 'asof_pitcher_success_rate' in df_proc.columns and 'asof_pitcher_n' in df_proc.columns:
        n = df_proc['asof_pitcher_n']
        curr = df_proc['asof_pitcher_success_rate']
        df_proc['smoothed_pitcher_success_rate'] = (n * curr + C * prior_mean) / (n + C)
    return df_proc


def step8_batter_toughness_features(df):
    df_proc = df.copy()
    if 'asof_batter_success_rate' in df_proc.columns and 'asof_batter_middle_rate' in df_proc.columns:
        df_proc['tough_batter_index'] = (1.0 - df_proc['asof_batter_success_rate']) * (1.0 - df_proc['asof_batter_middle_rate'])
    return df_proc


def step9_garbage_time_features(df):
    df_proc = df.copy()
    df_proc['is_garbage_time'] = np.where(df_proc['score_diff_pitcher_team'].abs() >= 7, 1.0, 0.0)
    df_proc['garbage_time_index'] = df_proc['score_diff_pitcher_team'].abs() / (10 - df_proc['inning']).clip(lower=1)
    return df_proc


def step10_recent_form_momentum(df):
    df_proc = df.copy()
    tc = ['asof_pitcher_prev1_game_success_rate',
          'asof_pitcher_prev3_game_success_rate',
          'asof_pitcher_prev5_game_success_rate']
    if all(c in df_proc.columns for c in tc):
        p1, p3, p5 = df_proc[tc[0]], df_proc[tc[1]], df_proc[tc[2]]
        df_proc['momentum_short'] = p1 - p3
        df_proc['momentum_mid'] = p1 - p5
        df_proc['is_heating_up'] = np.where((p1 > p3) & (p3 > p5), 1.0, 0.0)
        df_proc['is_cooling_down'] = np.where((p1 < p3) & (p3 < p5), 1.0, 0.0)
    return df_proc


def step11_veteran_and_pressure_features(df):
    df_proc = df.copy()
    df_proc['is_rookie'] = np.where(df_proc['asof_pitcher_n'] < 684, 1.0, 0.0)
    df_proc['is_veteran'] = np.where(df_proc['asof_pitcher_n'] > 3725, 1.0, 0.0)
    li_filled = df_proc['li'].fillna(0)
    df_proc['rookie_crisis_risk'] = df_proc['is_rookie'] * li_filled
    df_proc['veteran_clutch_ability'] = df_proc['is_veteran'] * li_filled
    return df_proc


def step12_first_pitch_tendency(df):
    df_proc = df.copy()
    if 'asof_pitcher_fastball_rate' in df_proc.columns and 'asof_pitcher_strike_rate' in df_proc.columns:
        if 'is_first_pitch' in df_proc.columns:
            df_proc['first_pitch_fastball_strike_idx'] = (
                df_proc['is_first_pitch'] * df_proc['asof_pitcher_fastball_rate'] * df_proc['asof_pitcher_strike_rate'])
    return df_proc


def step13_sac_fly_threat(df):
    df_proc = df.copy()
    is_3b = df_proc['base_state'].astype(str).apply(lambda x: 1.0 if '3' in x else 0.0)
    df_proc['is_sac_fly_threat'] = np.where(
        (is_3b == 1.0) & (df_proc['outs_before'] < 2)
        & (df_proc['score_diff_pitcher_team'].abs() <= 3), 1.0, 0.0)
    return df_proc


def step14_convert_to_category(df):
    df_proc = df.copy()
    original_cat_cols = ['pitcher_id', 'batter_id', 'pitcher_team_id', 'batter_team_id',
                         'pitcher_hand', 'batter_hand', 'base_state', 'stadium',
                         'pitch_name', 'top_bottom', 'game_type']
    created_cat_cols = ['is_weekend_day_game', 'is_heat_wave_game', 'is_pure_starter',
                        'is_long_relief', 'is_short_relief', 'is_same_hand', 'is_first_pitch',
                        'is_full_count', 'count_advantage', 'is_waste_pitch_sit',
                        'is_must_strike_sit', 'is_risp', 'is_strict_inherited_runner',
                        'is_self_risp', 'is_steal_threat_sit', 'is_sac_fly_threat',
                        'is_garbage_time', 'is_rookie', 'is_veteran',
                        'is_heating_up', 'is_cooling_down']
    all_cat_cols = [c for c in original_cat_cols + created_cat_cols if c in df_proc.columns]
    for c in all_cat_cols:
        df_proc[c] = df_proc[c].astype('category')
    return df_proc
"""

exec(STEPS_SRC)
print("step1~14 정의 완료")


In [ ]:
MAPPING_SRC = r"""
# pitcher_id <-> pitcher_trackman_id 매핑 재구축.
# 주최측이 준 pitcher_id_mapping.csv 는 구종비율 하나로만 매칭돼 약 91%가 틀렸다
# (시즌간 일관성 1.9%, 2024 커버리지 28%). 여기서 다시 만든다.
#   1단계 팀   : (월 x 요일 x 공수) 63차원 투구량 프로파일 -> 헝가리안.
#                검증 = 10개 팀이 6시즌 내내 같은 프랜차이즈로 대응되는가 (10/10).
#                ※ 월 단위 9차원으로는 실패한다 - 팀별 월간 분포가 거의 같아 비용이 평평해진다.
#   2단계 투수 : 팀-시즌 안에서 등판 프로파일 + 이닝 분포 + 구종배합 + 총투구량. 손은 하드제약.
#                검증 = 교정 전 시즌간 일관성 90.9% (매칭에 시즌간 정보를 안 쓰므로 순환 아님).
# 이 문자열이 단일 소스다. tools/rebuild_pitcher_mapping.py 가 노트북에서 이걸 읽어 쓴다.
from scipy.optimize import linear_sum_assignment

_MINOR_PREFIX = ('MIN_', 'KBO_', 'ACE_')   # 2군 / 올스타 / 기타


def _mp_prep(train_df, trackman_df):
    tr = train_df[['season', 'game_month', 'game_dayofweek', 'inning', 'top_bottom',
                   'pitcher_id', 'pitcher_hand', 'pitcher_team_id', 'asof_pitcher_pitchmix_n',
                   'asof_pitcher_fastball_rate', 'asof_pitcher_breaking_rate',
                   'asof_pitcher_offspeed_rate']].copy()
    tm = trackman_df[['season', 'game_month', 'game_dayofweek', 'inning', 'top_bottom',
                      'pitcher_trackman_id', 'pitcher_hand', 'pitcher_team',
                      'pitch_type_group']].copy()
    # 손 코딩이 다르다: train 은 1=Left/2=Right 정수, trackman 은 'Left'/'Right' 문자열
    tr['pitcher_hand'] = tr['pitcher_hand'].map({1: 'L', 2: 'R'})
    tm['pitcher_hand'] = tm['pitcher_hand'].map({'Left': 'L', 'Right': 'R'})
    tr['tb'] = tr['top_bottom']
    tm['tb'] = tm['top_bottom'].map({'Top': 'T', 'Bottom': 'B'})
    tm['grp'] = tm['pitch_type_group'].astype(str).str.lower()
    tm['team'] = tm['pitcher_team'].replace({'SK_WYV': 'SSG_LAN'})   # 2021 개명, 같은 프랜차이즈
    tm['is_major'] = ~tm['pitcher_team'].str.startswith(_MINOR_PREFIX, na=False)
    return tr, tm


def _mp_cells(df, key):
    d = df.assign(c=df['game_month'].astype(str) + '_' +
                    df['game_dayofweek'].astype(str) + '_' + df['tb'])
    return d.pivot_table(index=key, columns='c', aggfunc='size', fill_value=0).astype(float)


def _mp_unit(X):
    return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-9)


def _mp_match_teams(tr, tm, seasons):
    major = tm[tm['is_major']]
    rows = []
    for s in seasons:
        pa = _mp_cells(tr[tr['season'] == s], 'pitcher_team_id')
        pb = _mp_cells(major[major['season'] == s], 'team')
        pa, pb = pa.div(pa.sum(1), axis=0), pb.div(pb.sum(1), axis=0)
        cols = sorted(set(pa.columns) & set(pb.columns))
        A, B = pa[cols].values, pb[cols].values
        C = ((A[:, None, :] - B[None, :, :]) ** 2).sum(-1)
        r, c = linear_sum_assignment(C)
        rows += [dict(season=s, tid=pa.index[i], code=pb.index[j]) for i, j in zip(r, c)]
    piv = pd.DataFrame(rows).pivot(index='tid', columns='season', values='code')
    stable = int((piv.nunique(axis=1) == 1).sum())
    print(f"  [팀] 6시즌 내내 동일 프랜차이즈: {stable}/{len(piv)}")
    if stable != len(piv):
        raise RuntimeError("팀 매칭이 시즌 간 불일치.\n" + piv.to_string())
    return piv.iloc[:, 0].to_dict()


def _mp_train_mix(sub):
    '''train 의 누적 asof 비율에서 그 시즌만의 구종배합을 복원'''
    g = sub.sort_values('asof_pitcher_pitchmix_n').groupby('pitcher_id')
    n0 = g['asof_pitcher_pitchmix_n'].first()
    n1 = g['asof_pitcher_pitchmix_n'].last()
    out = {c: g[col].last() * n1 - g[col].first() * n0 for c, col in
           [('fastball', 'asof_pitcher_fastball_rate'),
            ('breaking', 'asof_pitcher_breaking_rate'),
            ('offspeed', 'asof_pitcher_offspeed_rate')]}
    M = pd.DataFrame(out)
    return M.div(M.sum(1).replace(0, np.nan), axis=0)


def build_pitcher_map(train_df, trackman_df):
    tr, tm = _mp_prep(train_df, trackman_df)
    seasons = sorted(tr['season'].unique())
    team_of = _mp_match_teams(tr, tm, seasons)
    tr = tr.assign(team=tr['pitcher_team_id'].map(team_of))
    major = tm[tm['is_major']]
    mixsrc = tm[tm['grp'].isin(['fastball', 'breaking', 'offspeed'])]  # 배합은 2군 포함
    MIX = ['fastball', 'breaking', 'offspeed']
    rows = []
    for s in seasons:
        a_all, b_all = tr[tr['season'] == s], major[major['season'] == s]
        mix_a = _mp_train_mix(a_all)
        ms = mixsrc[mixsrc['season'] == s]
        mix_b = pd.crosstab(ms['pitcher_trackman_id'], ms['grp'], normalize='index')
        for team in sorted(set(team_of.values())):
            a, b = a_all[a_all['team'] == team], b_all[b_all['team'] == team]
            if a.empty or b.empty:
                continue
            Pa, Pb = _mp_cells(a, 'pitcher_id'), _mp_cells(b, 'pitcher_trackman_id')
            Ia = a.assign(i=a['inning'].clip(1, 10)).pivot_table(
                index='pitcher_id', columns='i', aggfunc='size', fill_value=0
                ).reindex(columns=range(1, 11), fill_value=0).astype(float)
            Ib = b.assign(i=b['inning'].clip(1, 10)).pivot_table(
                index='pitcher_trackman_id', columns='i', aggfunc='size', fill_value=0
                ).reindex(columns=range(1, 11), fill_value=0).astype(float)
            cols = sorted(set(Pa.columns) & set(Pb.columns))
            ma = mix_a.reindex(Pa.index).reindex(columns=MIX).fillna(0.34).values
            mb = mix_b.reindex(Pb.index).reindex(columns=MIX).fillna(0.34).values
            ta, tb = Pa.values.sum(1), Pb.values.sum(1)
            c_sched = 1 - _mp_unit(Pa[cols].values) @ _mp_unit(Pb[cols].values).T
            c_inn = ((_mp_unit(Ia.values)[:, None, :] -
                      _mp_unit(Ib.values)[None, :, :]) ** 2).sum(-1)
            c_mix = ((ma[:, None, :] - mb[None, :, :]) ** 2).sum(-1)
            c_tot = (np.log1p(ta)[:, None] - np.log1p(tb)[None, :]) ** 2 * 0.05
            ha = a.groupby('pitcher_id')['pitcher_hand'].first().reindex(Pa.index).values
            hb = b.groupby('pitcher_trackman_id')['pitcher_hand'].first().reindex(Pb.index).values
            C = c_sched + c_inn + 2.0 * c_mix + c_tot + 100 * (ha[:, None] != hb[None, :])
            for i, j in zip(*linear_sum_assignment(C)):
                srt = np.sort(C[i])
                rows.append(dict(season=s, pitcher_id=Pa.index[i],
                                 pitcher_trackman_id=Pb.index[j], cost=C[i, j],
                                 margin=srt[1] - srt[0] if len(srt) > 1 else np.inf,
                                 n_tm=tb[j]))
    res = pd.DataFrame(rows)
    # 트레이드 선수는 여러 팀에서 후보가 나오므로 시즌별 1:1 로 정리
    best = res.sort_values('cost').groupby(['season', 'pitcher_id'], as_index=False).first()
    best = best.sort_values('cost').groupby(['season', 'pitcher_trackman_id'],
                                            as_index=False).first()
    vote = best.groupby(['pitcher_trackman_id', 'pitcher_id'])['n_tm'].sum().reset_index()
    win = (vote.sort_values('n_tm', ascending=False)
              .groupby('pitcher_trackman_id', as_index=False).first()
              .rename(columns={'pitcher_id': 'vote_pid'})[['pitcher_trackman_id', 'vote_pid']])
    best = best.merge(win, on='pitcher_trackman_id')
    # 검증은 반드시 다수결 '이전' 값으로. 교정 후에는 정의상 100%라 증거가 못 된다.
    g = best.groupby('pitcher_trackman_id')['pitcher_id']
    multi = g.nunique()[g.size() > 1]
    print(f"  [검증] 교정 전 시즌간 일관성 {(multi == 1).mean() * 100:.1f}% "
          f"(2시즌+ 등장 {len(multi)}명)")
    print(f"  [투수] 시즌간 다수결 교정 {int((best['pitcher_id'] != best['vote_pid']).sum())} "
          f"/ {len(best)}쌍")
    best['pitcher_id'] = best['vote_pid']
    out = best[['season', 'pitcher_id', 'pitcher_trackman_id', 'cost', 'margin']].copy()
    out['conf'] = np.where(out['cost'] <= out['cost'].quantile(0.75), 'high',
                    np.where(out['cost'] <= out['cost'].quantile(0.90), 'mid', 'low'))
    return out.sort_values(['season', 'pitcher_id']).reset_index(drop=True)
"""

exec(MAPPING_SRC)
print("build_pitcher_map 정의 완료")


In [ ]:
def step15_prep_trackman_data(trackman_df, pitcher_map_df):
    # 매핑에 season 이 있으면 반드시 시즌까지 키로 쓴다. pitcher_trackman_id 단독으로 붙이면
    # 한 투구가 여러 투수에게 중복 귀속돼 1.6배로 팽창한다 (2026-08-19 발견).
    keys = ['season', 'pitcher_trackman_id'] if 'season' in pitcher_map_df.columns \
        else ['pitcher_trackman_id']
    tm = pd.merge(trackman_df, pitcher_map_df[keys + ['pitcher_id']].drop_duplicates(),
                  on=keys, how='inner')
    if len(tm) > len(trackman_df):
        raise RuntimeError(f"트랙맨 병합이 팽창했습니다 ({len(trackman_df):,} -> {len(tm):,}). "
                           "매핑 키를 확인하세요.")
    b, s = tm['balls_before'], tm['strikes_before']
    p_ahead = ((b == 0) & (s == 1)) | ((b == 0) & (s == 2)) | ((b == 1) & (s == 2))
    b_ahead = ((b == 1) & (s == 0)) | ((b == 2) & (s == 0)) | ((b == 3) & (s == 0)) | ((b == 2) & (s == 1)) | ((b == 3) & (s == 1))
    neu = ((b == 1) & (s == 1)) | ((b == 2) & (s == 2))
    tm['count_advantage'] = np.select([p_ahead, b_ahead, neu],
                                       ['Pitcher', 'Batter', 'Neutral'], default='None')
    tm['pitch_group'] = tm['pitch_type_group'].astype(str).str.lower()
    return tm[tm['pitch_group'].isin(['fastball', 'breaking', 'offspeed'])].copy()


def step16_calc_expected_difficulty(tm):
    groups = ['fastball', 'breaking', 'offspeed']
    sit = tm.groupby(['season', 'game_month', 'pitcher_id', 'count_advantage', 'pitch_group']
                     ).size().unstack(fill_value=0).reset_index()
    for c in groups:
        if c not in sit.columns:
            sit[c] = 0
    sit = sit.sort_values(by=['pitcher_id', 'count_advantage', 'season', 'game_month'])
    g = sit.groupby(['pitcher_id', 'count_advantage'])
    sit['past_fb'] = g['fastball'].cumsum() - sit['fastball']
    sit['past_br'] = g['breaking'].cumsum() - sit['breaking']
    sit['past_off'] = g['offspeed'].cumsum() - sit['offspeed']
    tot = sit['past_fb'] + sit['past_br'] + sit['past_off']
    sit['past_total'] = tot
    sit['exp_fb_prob'] = np.where(tot > 0, sit['past_fb'] / tot, 0)
    sit['exp_br_prob'] = np.where(tot > 0, sit['past_br'] / tot, 0)
    sit['exp_off_prob'] = np.where(tot > 0, sit['past_off'] / tot, 0)

    dm = tm.groupby(['season', 'game_month', 'pitcher_id', 'pitch_group'])[['rel_height', 'rel_side']].std()
    dm['diff_score'] = dm['rel_height'] + dm['rel_side']
    dm = dm.reset_index()
    dp = dm.pivot_table(index=['season', 'game_month', 'pitcher_id'],
                        columns='pitch_group', values='diff_score', fill_value=np.nan).reset_index()
    for c in groups:
        if c not in dp.columns:
            dp[c] = 0
    dp = dp.sort_values(by=['pitcher_id', 'season', 'game_month'])
    gd = dp.groupby(['pitcher_id'])
    dp['past_fb_diff'] = gd['fastball'].transform(lambda x: x.shift(1).expanding().mean())
    dp['past_br_diff'] = gd['breaking'].transform(lambda x: x.shift(1).expanding().mean())
    dp['past_off_diff'] = gd['offspeed'].transform(lambda x: x.shift(1).expanding().mean())

    res = pd.merge(sit, dp, on=['season', 'game_month', 'pitcher_id'], how='left')
    res['expected_control_difficulty'] = (res['exp_fb_prob'] * res['past_fb_diff']
                                          + res['exp_br_prob'] * res['past_br_diff']
                                          + res['exp_off_prob'] * res['past_off_diff'])
    return res[['season', 'game_month', 'pitcher_id', 'count_advantage', 'expected_control_difficulty']]


def step17_calc_pitch_speed(tm):
    fb = tm[tm['pitch_group'] == 'fastball']
    sp = fb.groupby(['season', 'game_month', 'pitcher_id'])['rel_speed'].mean().reset_index()
    sp = sp.sort_values(by=['pitcher_id', 'season', 'game_month'])
    sp['past_fb_speed_mean'] = sp.groupby(['pitcher_id'])['rel_speed'].transform(
        lambda x: x.shift(1).expanding().mean())
    return sp[['season', 'game_month', 'pitcher_id', 'past_fb_speed_mean']]


def step18_calc_pitch_consistency_by_group(tm):
    groups = ['fastball', 'breaking', 'offspeed']
    metrics = ['rel_height_std', 'rel_side_std', 'extension_std',
               'spin_rate_std', 'vert_break_std', 'horz_break_std']
    cm = tm.groupby(['season', 'game_month', 'pitcher_id', 'pitch_group']).agg(
        rel_height_std=('rel_height', 'std'), rel_side_std=('rel_side', 'std'),
        extension_std=('extension', 'std'), spin_rate_std=('spin_rate', 'std'),
        vert_break_std=('induced_vert_break', 'std'), horz_break_std=('horz_break', 'std')
    ).reset_index()
    pv = cm.pivot_table(index=['season', 'game_month', 'pitcher_id'],
                        columns='pitch_group', values=metrics, fill_value=np.nan)
    pv.columns = [f"{grp}_{val}" for val, grp in pv.columns]
    pv = pv.reset_index()
    for pg in groups:
        for m in metrics:
            if f"{pg}_{m}" not in pv.columns:
                pv[f"{pg}_{m}"] = np.nan
    pv = pv.sort_values(by=['pitcher_id', 'season', 'game_month'])
    g = pv.groupby(['pitcher_id'])
    out_cols = ['season', 'game_month', 'pitcher_id']
    for pg in groups:
        for m in metrics:
            src, dst = f"{pg}_{m}", f"past_{pg}_{m}"
            pv[dst] = g[src].transform(lambda x: x.shift(1).expanding().mean())
            out_cols.append(dst)
    return pv[out_cols]


# ================= 릴리스 동역학 (2026-08-20 추가) =================
# 기존 step16~18 은 전부 (투수 x 월) 단위 표준편차라 세 가지가 한 숫자로 뭉개진다:
#   (a) 투구 간 기계적 흔들림  (b) 등판 간 드리프트  (c) 상황별 의도적 변화
# trackman_game_id / pitch_no 를 쓰면 분리할 수 있는데 여태 안 쓰고 있었다.
# 실제로 within(0.0304) 과 between(0.0274) 이 비슷한 크기 -> 절반이 다른 성분이었다.
# 검증: 2024 홀드아웃 6-seed 짝지어 +20(원본)/+22(재중심화).
#       절대점수 기준 신규 최저 736 > 기준선 평균 728 (기준선이 흔들려 차이 편차가 큼).

REL = ['rel_height', 'rel_side']


def build_release_dynamics(tm):
    """tm: step15 를 통과한 트랙맨 (pitcher_id 부착, 구종군 필터됨)"""
    t = tm.sort_values(['pitcher_id', 'trackman_game_id', 'pitch_no']).copy()
    out_key = ['season', 'game_month', 'pitcher_id', 'trackman_game_id']

    # --- 1) 연속 투구 간 릴리스 이동량 (같은 등판, 같은 구종군) ---
    g = t.groupby(out_key + ['pitch_group'], sort=False)
    t['seq_jump'] = np.sqrt(g['rel_height'].diff() ** 2 + g['rel_side'].diff() ** 2)

    # --- 2) 등판 단위 집계 ---
    agg = {'seq_jump': ('seq_jump', 'mean'), 'n': ('rel_height', 'size')}
    for c in REL + ['extension']:
        agg[f'w_{c}'] = (c, 'std')      # 등판 내 흔들림
        agg[f'm_{c}'] = (c, 'mean')     # 등판 중심 (등판 간 드리프트 계산용)
    outing = t.groupby(out_key, sort=False).agg(**agg).reset_index()
    outing = outing[outing.n >= 5]      # 5구 미만 등판은 통계가 무의미

    # --- 3) 등판 내 구속 감소 (fastball) ---
    fb = t[t.pitch_group == 'fastball'].copy()
    fb['rk'] = fb.groupby(out_key, sort=False).cumcount()
    fb['tot'] = fb.groupby(out_key, sort=False)['rk'].transform('size')
    fb = fb[fb.tot >= 9]
    fb['part'] = np.where(fb.rk < fb.tot / 3, 'early',
                   np.where(fb.rk >= 2 * fb.tot / 3, 'late', 'mid'))
    sp = fb[fb.part != 'mid'].pivot_table(index=out_key, columns='part',
                                          values='rel_speed', aggfunc='mean')
    sp['fb_speed_decay'] = sp.get('late', np.nan) - sp.get('early', np.nan)
    outing = outing.merge(sp[['fb_speed_decay']].reset_index(), on=out_key, how='left')

    # --- 4) 월 단위로 모으기: within 은 평균, between 은 등판중심의 표준편차 ---
    mkey = ['season', 'game_month', 'pitcher_id']
    m = outing.groupby(mkey).agg(
        seq_jump=('seq_jump', 'mean'),
        within_rel_h=('w_rel_height', 'mean'), within_rel_s=('w_rel_side', 'mean'),
        within_ext=('w_extension', 'mean'),
        between_rel_h=('m_rel_height', 'std'), between_rel_s=('m_rel_side', 'std'),
        between_ext=('m_extension', 'std'),
        fb_speed_decay=('fb_speed_decay', 'mean'),
        n_outing=('n', 'size'),
    ).reset_index()

    # --- 5) 터널링: 구종군 간 릴리스 중심 거리 ---
    cen = t.groupby(mkey + ['pitch_group'])[REL].mean().unstack('pitch_group')
    def gap(a, b):
        try:
            return np.sqrt((cen[('rel_height', a)] - cen[('rel_height', b)]) ** 2
                           + (cen[('rel_side', a)] - cen[('rel_side', b)]) ** 2)
        except KeyError:
            return pd.Series(np.nan, index=cen.index)
    tun = pd.DataFrame({'tunnel_fb_br': gap('fastball', 'breaking'),
                        'tunnel_fb_off': gap('fastball', 'offspeed')}).reset_index()
    m = m.merge(tun, on=mkey, how='left')

    # --- 6) 카운트 압박 하 릴리스 흔들림 차 ---
    cs = t.groupby(mkey + ['count_advantage'])[REL].std()
    cs = (cs['rel_height'] + cs['rel_side']).unstack('count_advantage')
    if 'Batter' in cs.columns and 'Pitcher' in cs.columns:
        m = m.merge((cs['Batter'] - cs['Pitcher']).rename('cnt_rel_gap').reset_index(),
                    on=mkey, how='left')
    else:
        m['cnt_rel_gap'] = np.nan

    # --- 7) leak-free 누적: 그 달 이전까지의 평균 ---
    cols = [c for c in m.columns if c not in mkey]
    m = m.sort_values(['pitcher_id', 'season', 'game_month'])
    gp = m.groupby('pitcher_id')
    for c in cols:
        m['past_' + c] = gp[c].transform(lambda x: x.shift(1).expanding().mean())
    return m[mkey + ['past_' + c for c in cols]]


# ================= 조건부 투수통계 (2026-08-18 추가) =================
# 설계: 성공률이 매 시즌 단조 하락(.565->.486)하므로 원시 성공률을 그대로 쓰면 과거 시즌의
#       높은 수준이 그대로 섞여 들어온다. 그래서 '그 시즌 리그평균 대비 편차'로 디트렌드한 뒤
#       0(=리그평균)으로 shrink 하는 경험적 베이즈 방식을 쓴다.
#       표본이 적은 조합일수록 자동으로 0에 가까워지므로 콜드스타트도 자연히 처리된다.
# 검증: 2024 홀드아웃 3-seed 짝지어 비교에서 기준선 대비 +20(원본)/+27(재중심화)
COND_SPECS = [
    (['pitcher_id'],                                    200, 'cond_p'),
    (['pitcher_id', 'count_advantage'],                 100, 'cond_pc'),
    (['pitcher_id', 'batter_hand'],                     100, 'cond_ph'),
    (['pitcher_id', 'batter_hand', 'count_advantage'],   50, 'cond_phc'),
]


def _add_dev(df):
    """control_success 를 '그 시즌 리그평균 대비 편차'로 변환 (드리프트 제거)."""
    lg = df.groupby('season')['control_success'].mean()
    return df['control_success'] - df['season'].map(lg)


def build_cond_table(src, keys, C, name):
    g = src.groupby(keys, observed=True)['_dev'].agg(['sum', 'count']).reset_index()
    g[name] = g['sum'] / (g['count'] + C)          # 0(리그평균)으로 shrink
    return g[keys + [name]]


def attach_cond_features(df):
    """학습용: 각 행은 '그 시즌보다 과거' 데이터로만 인코딩 -> leak-free.
    (배포 시 2025 test 가 2019~2024 로 인코딩되는 것과 동일한 규칙)"""
    df = df.copy()
    df['_dev'] = _add_dev(df)
    seasons = sorted(df['season'].unique())
    for keys, C, name in COND_SPECS:
        col = np.full(len(df), np.nan)
        for s in seasons:
            past = df[df['season'] < s]
            if len(past) == 0:
                continue
            t = build_cond_table(past, keys, C, name).set_index(keys)[name]
            cur = (df['season'] == s).values
            sl = df.loc[cur, keys]
            idx = pd.MultiIndex.from_frame(sl) if len(keys) > 1 else pd.Index(sl[keys[0]])
            col[cur] = t.reindex(idx).values
        df[name] = col
        print(f"  {name}: 결측 {np.isnan(col).mean()*100:.1f}% (첫 시즌 + 신규투수)")
    return df.drop(columns=['_dev'])


def build_all_cond_tables(df):
    """추론용: 학습 전 시즌을 다 써서 만든 최종 룩업 테이블."""
    d = df.copy()
    d['_dev'] = _add_dev(d)
    return {name: build_cond_table(d, keys, C, name) for keys, C, name in COND_SPECS}


COND_COLS = [name for _, _, name in COND_SPECS]


In [ ]:
def run_full_pipeline(train_df, trackman_df, pitcher_map, trackman_mode='asof'):
    print(f"파이프라인 시작 (trackman_mode={trackman_mode})...")
    df_proc = train_df.copy()

    df_proc = step1_basic_features(df_proc)
    df_proc = step2_pitcher_role_features(df_proc)
    df_proc = step3_matchup_features(df_proc)
    df_proc = step4_refined_count_features(df_proc)
    df_proc = step5_pitches_per_inning(df_proc)
    df_proc = step6_combined_runner_features(df_proc)

    prior_mean = float(df_proc['asof_pitcher_success_rate'].mean())
    print(f"  prior_mean = {prior_mean:.6f}")

    df_proc = step7_bayesian_smoothing(df_proc, prior_mean=prior_mean)
    df_proc = step8_batter_toughness_features(df_proc)
    df_proc = step9_garbage_time_features(df_proc)
    df_proc = step10_recent_form_momentum(df_proc)
    df_proc = step11_veteran_and_pressure_features(df_proc)
    df_proc = step12_first_pitch_tendency(df_proc)
    df_proc = step13_sac_fly_threat(df_proc)

    if 'count_advantage' not in df_proc.columns:
        b, s = df_proc['balls_before'], df_proc['strikes_before']
        p_ahead = ((b == 0) & (s == 1)) | ((b == 0) & (s == 2)) | ((b == 1) & (s == 2))
        b_ahead = ((b == 1) & (s == 0)) | ((b == 2) & (s == 0)) | ((b == 3) & (s == 0)) | ((b == 2) & (s == 1)) | ((b == 3) & (s == 1))
        neu = ((b == 1) & (s == 1)) | ((b == 2) & (s == 2))
        df_proc['count_advantage'] = np.select([p_ahead, b_ahead, neu],
                                                ['Pitcher', 'Batter', 'Neutral'], default='None')

    tm_base = step15_prep_trackman_data(trackman_df, pitcher_map)
    feat_diff = step16_calc_expected_difficulty(tm_base)
    feat_speed = step17_calc_pitch_speed(tm_base)
    feat_rp = step18_calc_pitch_consistency_by_group(tm_base)
    # 릴리스 동역학: 키가 feat_rp 와 같고 컬럼이 past_ 로 시작하므로 여기 합치면
    # 저장/추론/zip 로직이 수정 없이 그대로 따라온다.
    if USE_RELEASE_DYNAMICS:
        _dyn = build_release_dynamics(tm_base)
        _n0 = len(feat_rp)
        feat_rp = feat_rp.merge(_dyn, on=['season', 'game_month', 'pitcher_id'], how='outer')
        print(f"  릴리스 동역학 {len(_dyn):,}행 -> feat_rp {_n0:,} -> {len(feat_rp):,}행 "
              f"(신규 {len([c for c in _dyn.columns if c.startswith('past_')])}개)")
    rp_value_cols = [c for c in feat_rp.columns if c.startswith('past_')]

    if trackman_mode == 'asof':
        for f in [feat_diff, feat_speed, feat_rp]:
            f['time_idx'] = f['season'] * 100 + f['game_month']
            f.sort_values('time_idx', inplace=True)
        df_proc['time_idx'] = df_proc['season'] * 100 + df_proc['game_month']
        df_proc = df_proc.sort_values('time_idx')

        df_proc = pd.merge_asof(
            df_proc,
            feat_diff[['time_idx', 'pitcher_id', 'count_advantage', 'expected_control_difficulty']],
            on='time_idx', by=['pitcher_id', 'count_advantage'], direction='backward')
        df_proc = pd.merge_asof(
            df_proc, feat_speed[['time_idx', 'pitcher_id', 'past_fb_speed_mean']],
            on='time_idx', by='pitcher_id', direction='backward')
        df_proc = pd.merge_asof(
            df_proc, feat_rp[['time_idx', 'pitcher_id'] + rp_value_cols],
            on='time_idx', by='pitcher_id', direction='backward')
        df_proc = df_proc.drop(columns=['time_idx'])
    else:
        df_proc = pd.merge(df_proc, feat_diff,
                           on=['season', 'game_month', 'pitcher_id', 'count_advantage'], how='left')
        df_proc = pd.merge(df_proc, feat_speed,
                           on=['season', 'game_month', 'pitcher_id'], how='left')
        df_proc = pd.merge(df_proc, feat_rp,
                           on=['season', 'game_month', 'pitcher_id'], how='left')
        for c in ['expected_control_difficulty', 'past_fb_speed_mean'] + rp_value_cols:
            if c in df_proc.columns:
                df_proc[c] = df_proc[c].fillna(0)

    # 조건부 투수통계 (step14 이전에 붙여야 함: pitcher_id/count_advantage 가 아직 원시 dtype)
    cond_tables = {}
    if USE_COND_STATS:
        print("조건부 투수통계 생성...")
        df_proc = attach_cond_features(df_proc)
        cond_tables = build_all_cond_tables(df_proc)   # 추론용 최종 테이블(전 시즌)

    df_proc = step14_convert_to_category(df_proc)
    print("파이프라인 완료.")
    return df_proc.reset_index(drop=True), prior_mean, feat_diff, feat_speed, feat_rp, cond_tables


In [ ]:
import glob as _g, os as _o
_c = sorted(_g.glob("/kaggle/input/**/train.csv", recursive=True))
if not _c:
    raise RuntimeError("train.csv 를 못 찾음. /kaggle/input 내용: "
                       + str([p for p, _, _ in _o.walk("/kaggle/input")][:40]))
DATA_DIR = _o.path.dirname(_c[0])
print("DATA_DIR =", DATA_DIR, flush=True)
df_train = pd.read_csv(f"{DATA_DIR}/train.csv")
df_trackman = pd.read_csv(f"{DATA_DIR}/trackman_history.csv")
pitcher_id_mapping = build_pitcher_map(df_train, df_trackman)
df_processed = run_full_pipeline(df_train, df_trackman, pitcher_id_mapping,
                                 trackman_mode=TRACKMAN_MODE)[0]
print("프레임", df_processed.shape)


In [ ]:
# ---- 편향 b_Y 시계열 ----
import time
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression
from catboost import CatBoostClassifier

YEARS = [2021, 2022, 2023, 2024]
NF = 3                      # 현행 오프셋 셀과 동일
SEED0 = 42
V4 = {"learning_rate": 0.022831883708228414, "depth": 8, "l2_leaf_reg": 8.552069332567962,
      "bagging_temperature": 0.05636104060100738, "random_strength": 0.7731135614050382}
ITERS = 1000                # 현행 본 학습과 동일

target_col = 'control_success'
drop_cols = [target_col, 'row_id', 'pitcher_id', 'batter_id', 'time_idx'] + DEAD_FEATURES
FEATS = [c for c in df_processed.columns if c not in drop_cols]
X = df_processed[FEATS].copy()
for c in [c for c in X.columns if X[c].dtype.name in ('category', 'object')]:
    X[c] = X[c].astype(str).astype('category')
CF = [c for c in X.columns if X[c].dtype.name == 'category']
y = df_processed[target_col].copy()
season = df_processed['season'].to_numpy()
LG = pd.Series(y.to_numpy()).groupby(season).mean()
print("시즌별 실제 성공률:")
for s in sorted(LG.index):
    print(f"  {s}: {LG[s]:.4f}" + (f"  (하락 {LG[s]-LG[s-1]:+.4f})" if s - 1 in LG else ""))


def measure_bias(Y):
    """~Y-1 학습 -> Y 예측. 현행 오프셋 셀과 같은 3-fold + isotonic 구조."""
    tr, va = season <= Y - 1, season == Y
    Xh, yh = X[tr], y[tr]
    Xv = X[va]
    skf = StratifiedKFold(n_splits=NF, shuffle=True, random_state=SEED0)
    ps = []
    for ti, vi in skf.split(Xh, yh):
        p = dict(V4); p.update(iterations=ITERS, eval_metric='Logloss', task_type='GPU',
                               cat_features=CF, random_seed=SEED0, verbose=0)
        m = CatBoostClassifier(**p)
        m.fit(Xh.iloc[ti], yh.iloc[ti])
        iso = IsotonicRegression(out_of_bounds='clip').fit(
            m.predict_proba(Xh.iloc[vi])[:, 1], yh.iloc[vi])
        ps.append(iso.predict(m.predict_proba(Xv)[:, 1]))
    return float(np.mean(ps, axis=0).mean()), int(tr.sum())


ROWS = []
for Y in YEARS:
    t0 = time.time()
    mean_pred, ntr = measure_bias(Y)
    act = float(LG[Y]); drop = float(LG[Y] - LG[Y - 1])
    ROWS.append(dict(Y=Y, n=ntr, pred=mean_pred, act=act, b=mean_pred - act, d=drop))
    print(f"  Y={Y} 학습 {ntr:,} | 평균예측 {mean_pred:.4f} 실제 {act:.4f} "
          f"| b={mean_pred-act:+.4f} 하락={drop:+.4f} ({time.time()-t0:.0f}s)", flush=True)

R = pd.DataFrame(ROWS)
print(f"\n=== 편향 시계열 ===")
print(f"{'Y':<7}{'학습행':>11}{'평균예측':>10}{'실제':>9}{'b':>10}{'하락폭':>10}{'b/하락':>9}")
for _, r in R.iterrows():
    print(f"{int(r.Y):<7}{int(r.n):>11,}{r.pred:>10.4f}{r.act:>9.4f}"
          f"{r.b:>+10.4f}{r.d:>+10.4f}{r.b/r.d:>9.2f}")

# 2025 하락폭 예측 (리그 평균 선형 외삽) — 모형 B 에 필요
ss = sorted(LG.index)
d2025 = float(np.polyval(np.polyfit(ss, LG.loc[ss].values, 1), 2025) - LG[2024])
print(f"\n2025 예상 하락폭 {d2025:+.4f} (리그평균 선형외삽 -> {LG[2024]+d2025:.4f})")


def fit_A(sub):      # b ~ 연도
    c = np.polyfit(sub.Y.values, sub.b.values, 1)
    return lambda Y, d: float(np.polyval(c, Y)), c


def fit_B(sub):      # b ~ 하락폭 (절편 포함)
    c = np.polyfit(sub.d.values, sub.b.values, 1)
    return lambda Y, d: float(np.polyval(c, d)), c


def fit_C(sub):      # b = k x 하락폭 (원점 통과)
    k = float((sub.b.values * sub.d.values).sum() / (sub.d.values ** 2).sum())
    return lambda Y, d: k * d, np.array([k, 0.0])


print(f"\n=== 검증: 2021~2023 으로 b_2024 를 맞히는가 ===")
sub = R[R.Y <= 2023]
true_b24 = float(R[R.Y == 2024].b.iloc[0])
d24 = float(R[R.Y == 2024].d.iloc[0])
BEST = (1e9, None)
for nm, fn in [('A: b ~ 연도', fit_A), ('B: b ~ 하락폭', fit_B), ('C: b = k x 하락폭', fit_C)]:
    f, c = fn(sub)
    pred = f(2024, d24)
    err = abs(pred - true_b24)
    pts = err ** 2 / (R[R.Y == 2024].act.iloc[0] * (1 - R[R.Y == 2024].act.iloc[0])) * 1e5
    print(f"  {nm:<18} 예측 b_2024={pred:+.4f}  실제 {true_b24:+.4f}  "
          f"오차 {err:.4f}  (손실 {pts:>5.0f}점)")
    if err < BEST[0]:
        BEST = (err, nm, fn)
print(f"  현행 방식(b_2023 을 그대로 사용) 예측 {float(R[R.Y==2023].b.iloc[0]):+.4f}  "
      f"오차 {abs(float(R[R.Y==2023].b.iloc[0])-true_b24):.4f}")
print(f"  -> 최선: {BEST[1]}")

print(f"\n=== 2025 외삽 (네 점 전부 사용) ===")
act24 = float(R[R.Y == 2024].act.iloc[0])
u = act24 * (1 - act24)
cur_b = float(R[R.Y == 2024].b.iloc[0])
print(f"{'모형':<20}{'b_2025':>10}{'현행 대비':>11}{'현행이 틀릴 때 손실':>20}")
for nm, fn in [('A: b ~ 연도', fit_A), ('B: b ~ 하락폭', fit_B), ('C: b = k x 하락폭', fit_C)]:
    f, c = fn(R)
    b25 = f(2025, d2025)
    diff = cur_b - b25
    print(f"{nm:<20}{b25:>+10.4f}{diff:>+11.4f}{diff**2/u*1e5:>19,.0f}점")
print(f"{'현행 (b_2024 재사용)':<20}{cur_b:>+10.4f}")

print(f"\n해석: 검증에서 이긴 모형의 b_2025 와 현행 b_2024 의 차이가 곧 얻거나 잃을 점수다.")
print(f"      차이가 0.005 미만이면 현행 유지, 0.010 이상이면 바꿀 값어치가 있다.")
